# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking task. The editor has limited time, so instead of giving them a binary label like “review/do not review,” it’s more useful to provide an ordered list ranging from the page that needs the most review to the one that needs the least. 
That’s why I chose ‘ranking’ instead of “classification.”

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I’m estimating/ranking: ctr_gap — how far the page lags behind the average CTR for its own position tier.
This is a proxy — not an actual, observed “this page is bad” label, but a rule I’ve defined (tier average − actual CTR).
In the real world, I don’t have an observed label like “an editor looked at this page and it was truly problematic,” so I’m working with this proxy and clearly stating that in the note.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

My metric for success: Precision@20 — an editor can realistically review about 20 pages per week, so I’ll measure how many of the top 20 pages on my ranked list are actually “worth reviewing” (high ctr_gap). For example, Precision@20 = 0.80 means that 16 of the first 20 pages are actually underperforming — meaning 80% of the editor’s time was spent in the right places.

Why I chose this: Accuracy (overall accuracy) would be misleading here because most pages already show “normal” performance—the model would achieve high accuracy even if it labeled everything as “normal.” Precision@20, on the other hand, directly measures the quality of the list the editor will actually use.

In [14]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

eligible = df[df['impressions_90d'] >= 100].copy()

expected_ctr = eligible.groupby('position_tier')['ctr'].mean()
eligible['expected_ctr'] = eligible['position_tier'].map(expected_ctr)
eligible['ctr_gap'] = eligible['expected_ctr'] - eligible['ctr']

print(eligible.shape)

(22006, 46)


In [15]:
# En yüksek ctr_gap'e sahip ilk 20 sayfa = "flag'lenen" liste
top_20 = eligible.sort_values('ctr_gap', ascending=False).head(20)
print(top_20[['content_id', 'position_tier', 'ctr', 'expected_ctr', 'ctr_gap']])
print()
print("The average ctr_gap for these 20 pages:", top_20['ctr_gap'].mean())
print("The average ctr_gap for all eligible pages:”", eligible['ctr_gap'].mean())

                 content_id position_tier  ctr  expected_ctr  ctr_gap
8939   content_478f26883850        page_1  0.0       0.35476  0.35476
29977  content_c87291853cab        page_1  0.0       0.35476  0.35476
29983  content_6880eb215048        page_1  0.0       0.35476  0.35476
8873   content_268b56dc0221        page_1  0.0       0.35476  0.35476
8915   content_e0ae71489787        page_1  0.0       0.35476  0.35476
33     content_d87a116e2c79        page_1  0.0       0.35476  0.35476
2282   content_3d2f81868b16        page_1  0.0       0.35476  0.35476
24588  content_dc0d776ba8e5        page_1  0.0       0.35476  0.35476
11665  content_50c7fe0d308f        page_1  0.0       0.35476  0.35476
11673  content_6b5196195b13        page_1  0.0       0.35476  0.35476
11677  content_a8ed3dfdc294        page_1  0.0       0.35476  0.35476
11682  content_ead17e13e806        page_1  0.0       0.35476  0.35476
11578  content_e6be4a754962        page_1  0.0       0.35476  0.35476
24737  content_d86bc

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

My unit of analysis: one page (content_id). Each row represents a 90-day performance summary for a single web page.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (“flag if CTR < 10%”) won’t work because “low CTR” means a different number for each position tier—an average of 35% on page_1 and 5% in deep. If I use a fixed threshold, I’ll unfairly flag pages on page_1 and miss the real issues in deep.

My `ctr_gap` rule partially solves this, but it still relies on a single signal (position). Manually testing with `if` statements to see if other factors—such as `main_intent`, `word_count`, or `content_type`—affect CTR isn’t practical—there are simply too many combinations. A model can learn these interactions from the data.

My plan: Keep `ctr_gap` as the baseline, train a model, and prove that the model’s Precision@20 is truly better than the baseline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.